In [1]:
import time
import random
import datetime
import urllib
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
#load stream_simulation_data
path ='/Users/sivhoung/Fraud_detection/stream_simulation_data/part-00000-955a79b5-98bb-4146-838c-ed7dd5a9c43b-c000.csv'
test_df = pd.read_csv(path)

### Database connection configuration
#### Load_data

In [3]:
#connecting
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"  # 如果你本地检查出来是18，请改为18
    "SERVER=paysim-db.c78a2mamcvtr.ap-southeast-2.rds.amazonaws.com,1433;"
    "DATABASE=FraudDetectionDB;"
    "UID=admin;"
    "PWD=GCTFFh6bOvcZMPSPmE2k;"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# 使用较小的连接池，专供流式持续 INSERT 使用
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", pool_size=5, max_overflow=10)

### Simulating bank streaming emitter active

In [4]:
print("Simulating live online banking transaction feeds to AWS RDS...")

Simulating live online banking transaction feeds to AWS RDS...


In [5]:
transaction_counter = 0

try:
    while True:
        # 3.1 随机模拟突发流量：每次随机生成 1 到 4 笔转账流水
        batch_size = random.randint(1, 4)
        samples = test_df.sample(batch_size)
        
        # 3.2 捕获当前绝对精确的本地系统时间（这才是真正的实时流时间戳！）
        current_real_time = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        
        # 3.3 使用原生的数据库连接执行插入，确保毫秒级写入
        with engine.begin() as connection:
            for idx, row in samples.iterrows():
                query = text("""
                    INSERT INTO dbo.live_transactions (
                        step, [type], amount, nameOrig, oldbalanceOrg, newbalanceOrig, 
                        nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud, transaction_timestamp
                    ) VALUES (
                        :step, :type, :amount, :nameOrig, :oldbalanceOrg, :newbalanceOrig, 
                        :nameDest, :oldbalanceDest, :newbalanceDest, :isFraud, :isFlaggedFraud, :transaction_timestamp
                    )
                """)
                
                # 安全性处理：如果列名真的不存在（比如被提前删了），则自动赋予默认安全数值，绝不卡死报错
                connection.execute(query, {
                    "step": int(row.get('step', 1)), # 如果没有 step 列，默认补成 1
                    "type": str(row.get('type', 'TRANSFER')),
                    "amount": float(row.get('amount', 0.0)),
                    "nameOrig": str(row.get('nameorig', 'UNKNOWN')),
                    "oldbalanceOrg": float(row.get('oldbalanceorg', 0.0)),
                    "newbalanceOrig": float(row.get('newbalanceorig', 0.0)),
                    "nameDest": str(row.get('namedest', 'UNKNOWN')),
                    "oldbalanceDest": float(row.get('oldbalancedest', 0.0)),
                    "newbalanceDest": float(row.get('newbalancedest', 0.0)),
                    "isFraud": int(row.get('isfraud', 0)),
                    "isFlaggedFraud": int(row.get('isflaggedfraud', 0)),
                    "transaction_timestamp": current_real_time
                })
                
                transaction_counter += 1
                
        print(f"[{current_real_time}] ⚡ Simulated Bank Stream Burst: Pushed {batch_size} transactions up to AWS. Total: {transaction_counter}")
        
        # 3.4 模拟真实的银行流水间隔频率（每隔 1.0 到 1.5 秒震荡发射一次）
        time.sleep(random.uniform(1.0, 1.5))

except KeyboardInterrupt:
    print("\n Streaming Data Simulation stopped safely by operator.")

[2026-06-19 15:24:17] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 4
[2026-06-19 15:24:50] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 8
[2026-06-19 15:24:53] ⚡ Simulated Bank Stream Burst: Pushed 1 transactions up to AWS. Total: 9
[2026-06-19 15:24:54] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 13
[2026-06-19 15:24:57] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 17
[2026-06-19 15:25:00] ⚡ Simulated Bank Stream Burst: Pushed 2 transactions up to AWS. Total: 19
[2026-06-19 15:25:02] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 23
[2026-06-19 15:25:05] ⚡ Simulated Bank Stream Burst: Pushed 4 transactions up to AWS. Total: 27
[2026-06-19 15:25:07] ⚡ Simulated Bank Stream Burst: Pushed 2 transactions up to AWS. Total: 29
[2026-06-19 15:25:10] ⚡ Simulated Bank Stream Burst: Pushed 2 transactions up to AWS. Total: 31
[2026-06-19 15:25:12] ⚡ Simulated Bank Stre

OperationalError: (pyodbc.OperationalError) ('08S01', '[08S01] [Microsoft][ODBC Driver 18 for SQL Server]TCP Provider: Error code 0x274C (10060) (SQLExecDirectW)')
[SQL: 
                    INSERT INTO dbo.live_transactions (
                        step, [type], amount, nameOrig, oldbalanceOrg, newbalanceOrig, 
                        nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud, transaction_timestamp
                    ) VALUES (
                        ?, ?, ?, ?, ?, ?, 
                        ?, ?, ?, ?, ?, ?
                    )
                ]
[parameters: (1, 'PAYMENT', 47874.64, 'C2105467227', 7776.0, 0.0, 'M99508135', 0.0, 0.0, 0, 0, '2026-06-19 15:40:26')]
(Background on this error at: https://sqlalche.me/e/20/e3q8)